### Packages

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Gephi clustering settings:
- import graph object
- Statistics: Connected Components: Run (standard settings)
- Statistics: Modularity: Run (standard settings)
- Layout: Circle Pack Layout: Hierarchy1 = Component ID, Hierarchy2 = Modularity Class

### Load data

Load all epitope obs dfs

In [2]:
from pathlib import Path

folder = Path("./sc_obs_files/epitopes")

df_list = [
    pd.read_csv(file)
    for file in folder.glob("*.csv")
]
qc_file = pd.read_excel('./ref_tcr_data/Library_QC.xlsx')

In [3]:
df_obs_new = pd.concat(df_list, ignore_index=True)

In [4]:
df_obs_new['donor_id'].nunique()

579

In [5]:
def process_df_obs(df):
    df = df.copy()
    # Rename TRBV10 -> TRBV4 wrong annotation by parse/cellranger
    df = df.replace("TRBV10", "TRBV4", regex=False)
    # Remove unwanted V genes: non-functional
    df = df[
        ~df["IR_VDJ_1_v_call"].isin(["TRBV21", "TRBV8"])
    ]
    # Keep only desired chain pairings - only single chain TCRs are considered
    df = df[
        df["chain_pairing"].isin([
            "single pair",
            "assigned_single_pair"
        ])
    ]
    return df.reset_index(drop=True).copy()

In [6]:
df_obs_new = process_df_obs(df_obs_new)
df_obs_new['tcr_id'] = df_obs_new['TCR']
df_obs_new['clone_size_mouse'] = df_obs_new['TCR_expansion']

In [7]:
qc_list = qc_file.tcr_id.tolist()
df_obs_new = df_obs_new[~df_obs_new.tcr_id.isin(qc_list)].reset_index(drop=True).copy()

In [8]:
df_obs_new['donor_id'].nunique()

579

## QC sequencing specificity annotation

Multi specificity annotations per tcr id are only epitope + below_threshold -> consistent annotation across experiments

In [9]:
multi_spec_values = (
    df_obs_new.groupby("tcr_id")["sequencing_specificity"]
    .agg(lambda x: sorted(set(x.dropna())))
)

multi_spec_values = multi_spec_values[
    multi_spec_values.apply(len) > 1
]


check = (
    df_obs_new.loc[df_obs_new["sequencing_specificity"] != "below_threshold"]
    .groupby("tcr_id")["sequencing_specificity"]
    .nunique()
)

print(check[check > 1])

Series([], Name: sequencing_specificity, dtype: int64)


In [10]:
specificity_map = (
    df_obs_new.loc[df_obs_new["sequencing_specificity"] != "below_threshold"]
    .groupby("tcr_id")["sequencing_specificity"]
    .first()
)

df_obs_new.loc[df_obs_new["tcr_id"].isin(specificity_map.index),"sequencing_specificity"] = df_obs_new.loc[
    df_obs_new["tcr_id"].isin(specificity_map.index), "tcr_id"].map(specificity_map)

Annotate sharedness

In [11]:
df_obs_new["shared_mouse"] = (
    df_obs_new.groupby("tcr_id")["donor_id"]
      .transform("nunique")
)

In [12]:
cols_to_keep = ['is_cell',
 'high_confidence',
 'multi_chain',
 'extra_chains',
 'IR_VJ_1_c_call',
 'IR_VDJ_1_c_call',
 'IR_VJ_1_d_call',
 'IR_VDJ_1_d_call',
 'IR_VJ_1_j_call',
 'IR_VDJ_1_j_call',
 'IR_VJ_1_junction',
 'IR_VDJ_1_junction',
 'IR_VJ_1_junction_aa',
 'IR_VDJ_1_junction_aa',
 'IR_VJ_1_v_call',
 'IR_VDJ_1_v_call',
 'has_ir',
 'experiment',
 'group',
 'multimer',
 'batch',
 'mouse',
 'mouse_concat',
 'isolated_specificity',
 'receptor_type',
 'receptor_subtype',
 'chain_pairing',
 'tra_id',
 'trb_id',
 'tcr_id',
 'TCR_expansion',
 'clonotype_size_mouse',
 'clone_size_mouse',
 'sequencing_specificity',
 'shared_mouse',
 'NAME',
 'donor_id',
 'biosample_id',
 'species',
 'species__strain',
 'species__ontology_label',
 'disease',
 'disease__strain',
 'disease__ontology_label',
 'disease__epitope',
 'disease__antigen',
 'disease__timepoint',
 'organ',
 'organ__ontology_label',
 'library_preparation_protocol',
 'library_preparation_protocol__ontology_label',
 'cell_type',
 'cell_type__ontology_label',
 'sex',
 'assigned_TRA',
 'assigned_TRB',
 'TRB',
 'TRA',
 'TCR',
 'TCR_expansion_full_data',
 'n_cells_mouse',
 'exp_group',
 'mean_tcr_activation_score',
 'cell_id']

In [13]:
df_obs_new = df_obs_new[cols_to_keep].copy()

### Make TCR data sheet

In [14]:
v_info = pd.read_csv('./10X_mouse_vdj/cellranger_mouse_v_ref.csv')
j_info = pd.read_csv('./10X_mouse_vdj/cellranger_mouse_j_ref.csv')

In [15]:
df = df_obs_new.rename(columns={
    "IR_VJ_1_junction_aa": "cdr3a_aa",
    "IR_VDJ_1_junction_aa": "cdr3b_aa",
    "IR_VJ_1_junction": "cdr3a_nt",
    "IR_VDJ_1_junction": "cdr3b_nt",
    "IR_VJ_1_v_call": "va",
    "IR_VJ_1_j_call": "ja",
    "IR_VJ_1_c_call": "ca",
    "IR_VDJ_1_v_call": "vb",
    "IR_VDJ_1_j_call": "jb",
    "IR_VDJ_1_d_call": "db",
    "IR_VDJ_1_c_call": "cb",
})

In [16]:
from preprocess_tcr_2 import stitch_nt_from_cellranger
from pandas.api.types import is_numeric_dtype

agg_dict = {}

for col in df.columns:
    #if col == "tcr_id":
    #    continue

    if is_numeric_dtype(df[col]):
        agg_dict[col] = "max"
    else:
        agg_dict[col] = "first"

df = (
    df.groupby("tcr_id", as_index=False, observed=True)
      .agg(agg_dict)
)
    
df = df.reset_index(drop=True)
df = stitch_nt_from_cellranger(
        df=df,
        v_info=v_info,
        j_info=j_info,
        v_gene_col="vb",
        j_gene_col="jb",
        cdr3_col="cdr3b_nt",
        c_col="cb",
        chain="beta"
        )
    
df = stitch_nt_from_cellranger(
        df=df,
        v_info=v_info,
        j_info=j_info,
        v_gene_col="va",
        j_gene_col="ja",
        cdr3_col="cdr3a_nt",
        c_col="ca",
        chain="alpha"
        )

df = df.reset_index(drop=True)
  

C:\Users\wwspa\miniconda3\envs\tcr_scripts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
df.columns

Index(['is_cell', 'high_confidence', 'multi_chain', 'extra_chains', 'ca', 'cb',
       'IR_VJ_1_d_call', 'db', 'ja', 'jb', 'cdr3a_nt', 'cdr3b_nt', 'cdr3a_aa',
       'cdr3b_aa', 'va', 'vb', 'has_ir', 'experiment', 'group', 'multimer',
       'batch', 'mouse', 'mouse_concat', 'isolated_specificity',
       'receptor_type', 'receptor_subtype', 'chain_pairing', 'tra_id',
       'trb_id', 'tcr_id', 'TCR_expansion', 'clonotype_size_mouse',
       'clone_size_mouse', 'sequencing_specificity', 'shared_mouse', 'NAME',
       'donor_id', 'biosample_id', 'species', 'species__strain',
       'species__ontology_label', 'disease', 'disease__strain',
       'disease__ontology_label', 'disease__epitope', 'disease__antigen',
       'disease__timepoint', 'organ', 'organ__ontology_label',
       'library_preparation_protocol',
       'library_preparation_protocol__ontology_label', 'cell_type',
       'cell_type__ontology_label', 'sex', 'assigned_TRA', 'assigned_TRB',
       'TRB', 'TRA', 'TCR', 'TCR_exp

### Annotate prediction scores

In [18]:
df_pred = pd.read_csv('./ref_tcr_data/HELMHOLTZ_MOUSE_TCR_ANY_EPITOPE_single_TCRs_scores.csv')

In [19]:
df_pred.rename(columns={"prediction_siinfekl_reactive": "prediction_ova",
                             "prediction_siinfekl_reactive_split_0": "prediction_ova_0",
                             "prediction_siinfekl_reactive_split_1": "prediction_ova_1",
                             "prediction_siinfekl_reactive_split_2": "prediction_ova_2",
                             "prediction_siinfekl_reactive_split_3": "prediction_ova_3",
                             "prediction_siinfekl_reactive_split_4": "prediction_ova_4",
                            }, inplace=True)
#tcr_list = df_pred.tcr_id.tolist()
#df = df[df.tcr_id.isin(tcr_list)].reset_index(drop=True).copy()

In [20]:
col_list = ['prediction_GP33', 'prediction_M45', 'prediction_ova']

In [21]:
for col in col_list: 
    df[col] = (
        df["tcr_id"]
        .map(
            df_pred.set_index("tcr_id")[col]
        )
    )


In [22]:
pred_cols = ["prediction_GP33", "prediction_M45", "prediction_ova"]
values = ["GP33", "M45", "OVA"]
threshold = 0.5

col_to_val = dict(zip(pred_cols, values))

# Boolean mask of predictions > threshold
mask = df[pred_cols] > threshold
has_prediction = mask.any(axis=1)

# Initialize explicitly to avoid stale values
df["annotated_specificity"] = "below_threshold"
df["predicted_specificity"] = "below_threshold"

# Assign single or multi-hit predictions
df.loc[has_prediction, "predicted_specificity"] = (
    mask.loc[has_prediction]
    .apply(
        lambda row: "_".join([col_to_val[c] for c in pred_cols if row[c]]),
        axis=1,
    )
)

# Start annotation from seq specificity
df["annotated_specificity"] = df["sequencing_specificity"].copy()

# Multi-epitope predictions are considered below threshold / ambiguous
multi_prediction = (
    df["predicted_specificity"].str.contains("_", na=False)
    & (df["predicted_specificity"] != "below_threshold")
)

df.loc[multi_prediction, "annotated_specificity"] = "below_threshold"

# Prediction must match isolated specificity
mismatch = df["annotated_specificity"] != df["isolated_specificity"]

df.loc[mismatch, "annotated_specificity"] = "below_threshold"

# For sufficiently large clones, trust isolation method
large_clone = df["clone_size_mouse"] > 2

df.loc[
    (df["annotated_specificity"] == "below_threshold") & large_clone,
    "annotated_specificity"
] = df.loc[
    (df["annotated_specificity"] == "below_threshold") & large_clone,
    "isolated_specificity"
]

## Annotate pgen

In [23]:
from preprocess_tcr_2 import format_for_tcrdist

df['clonotype_origin'] = df['tcr_id']
df_dist = format_for_tcrdist(df)

from preprocess_tcr_2 import run_tcrdist_pgen_only

# Assume you already have:
#   df_clonotypes_all  (output of the previous function)
#   df_cells           (original per-cell DataFrame)
#   metrics_a, metrics_b, weights_a, weights_b, kargs_a, kargs_b
result_list = []

results = run_tcrdist_pgen_only(
        df_clonotypes_all = df_dist, ### dataframe for TCRdist calculation with formatting of v genes and clone id and so on
        df_original       = df, ### dataframe before it was processed for TCRdist calculation
        db_file           = "alphabeta_gammadelta_db.tsv",
        olga_beta_folder  = "mouse_T_beta",
        olga_alpha_folder = "mouse_T_alpha",
        cpus              = 12
    )

Generate new TR file


C:\Users\wwspa\miniconda3\envs\tcr_scripts\Lib\site-packages\tcrdist\repertoire.py:500: UserWarning: TRAV12-4*01 gene was not recognized in reference db no cdr seq could be inferred
  f0 = lambda v : self._map_gene_to_reference_seq2(gene = v,
C:\Users\wwspa\miniconda3\envs\tcr_scripts\Lib\site-packages\tcrdist\repertoire.py:504: UserWarning: TRAV12-4*01 gene was not recognized in reference db no cdr seq could be inferred
  f1 = lambda v : self._map_gene_to_reference_seq2(gene = v,
C:\Users\wwspa\miniconda3\envs\tcr_scripts\Lib\site-packages\tcrdist\repertoire.py:508: UserWarning: TRAV12-4*01 gene was not recognized in reference db no cdr seq could be inferred
  f2 = lambda v : self._map_gene_to_reference_seq2(gene = v,
C:\Users\wwspa\miniconda3\envs\tcr_scripts\Lib\site-packages\tcrdist\repertoire.py:190: UserWarning: Not all cells/sequences could be grouped into clones.27 of 232775 were not captured. This occurs when any of the values in the index columns are null or missing for a giv

calculating TCR pgens


62640it [02:22, 438.95it/s]                                                                                            
62640it [01:28, 704.84it/s]                                                                                            


Merging data


In [24]:
df_pgen = results[0].copy()
df_pgen['tcr_id'] = df_pgen['clonotype_origin']
df["pgen_cdr3_b_aa"] = (
    df["tcr_id"]
    .map(
        df_pgen.set_index("tcr_id")["pgen_cdr3_b_aa"]
    )
)
df["pgen_cdr3_a_aa"] = (
    df["tcr_id"]
    .map(
        df_pgen.set_index("tcr_id")["pgen_cdr3_a_aa"]
    )
)
df["pgen_tcr"] = df["pgen_cdr3_a_aa"] * df["pgen_cdr3_b_aa"]

## Annotate obs df

In [25]:
df.annotated_specificity.value_counts()

annotated_specificity
below_threshold    43963
OVA                10372
GP33                5313
M45                 2998
Name: count, dtype: int64

In [26]:
col_list = ['prediction_GP33', 'prediction_M45', 'prediction_ova',
            'annotated_specificity', 'predicted_specificity', 'sequencing_specificity']

In [27]:
for col in col_list: 
    df_obs_new[col] = (
        df_obs_new["tcr_id"]
        .map(
            df.set_index("tcr_id")[col]
        )
    )


In [28]:
df_tcr = df
df_obs = df_obs_new

#annotate repertoire size
df_obs['n_cells_mouse'] = df_obs.groupby('donor_id')['donor_id'].transform('count')
df_obs['n_TCRs_mouse'] = df_obs.groupby('donor_id')['TCR'].transform('nunique')

reactive_mask = df_obs["annotated_specificity"] != "below_threshold"

# Number of reactive cells per mouse
donor_sizes = (
    df_obs.loc[reactive_mask]
    .groupby("donor_id")
    .size()
    .rename("donor_size")
)

# Mean reactive repertoire size across mice
mean_size = donor_sizes.mean()

# Size factor per mouse
size_factors = (
    donor_sizes / mean_size
).rename("size_factor")

# Optional summary table
df_size_factors = (
    size_factors
    .reset_index()
)

# Annotate size factor to all cells
df_obs["size_factor"] = df_obs["donor_id"].map(size_factors)

# Normalize clone sizes
# Mice without reactive cells retain their original clone sizes
df_obs["norm_TCR_expansion"] = (
    df_obs["TCR_expansion"] /
    df_obs["size_factor"].fillna(0)
) + 1

#annotate normalized expansion in tcr_df
norm_df = pd.DataFrame(df_obs.groupby('TCR').agg(
    norm_TCR_expansion = ('norm_TCR_expansion','max'),)).reset_index()
norm_df.rename(columns={'TCR':'tcr_id'}, inplace = True)

df_tcr["norm_TCR_expansion"] = (
    df_tcr["tcr_id"]
    .map(
        norm_df.set_index("tcr_id")["norm_TCR_expansion"]
    )
    .fillna(df_tcr["TCR_expansion"])
)

In [29]:
print(len(df_obs))
obs_df_list = df_obs[df_obs.annotated_specificity!='below_threshold'].tcr_id.unique().tolist()
tcr_df_list = df[df.annotated_specificity!='below_threshold'].tcr_id.tolist()
missing = [tcr for tcr in obs_df_list if tcr not in tcr_df_list]
df_obs = df_obs[~df_obs.tcr_id.isin(missing)].reset_index(drop=True).copy()
print(len(df_obs))

279578
279578


Export TCR data sheet

In [30]:
df_backup = df.copy()

In [31]:
df.to_excel('./processed_tcr_data_new_sc_processing/TCR_data_busch_lab_ova_gp33_m45_all_tcrs.xlsx')
df.to_csv('./processed_tcr_data_new_sc_processing/TCR_data_busch_lab_ova_gp33_m45_all_tcrs.csv')

In [32]:
df = df[df.annotated_specificity.isin(['OVA','GP33','M45'])].reset_index(drop=True).copy()

In [33]:
df.to_excel('./processed_tcr_data_new_sc_processing/TCR_data_busch_lab_ova_gp33_m45_all_tcrs_reactive.xlsx')
df.to_csv('./processed_tcr_data_new_sc_processing/TCR_data_busch_lab_ova_gp33_m45_all_tcrs_reactive.csv')

In [34]:
df_obs_backup = df_obs.copy()

In [35]:
df_obs.to_excel('./processed_tcr_data_new_sc_processing/TCR_obs_data_busch_lab_ova_gp33_m45_all_tcrs.xlsx')
df_obs.to_csv('./processed_tcr_data_new_sc_processing/TCR_obs_data_busch_lab_ova_gp33_m45_all_tcrs.csv')
df_obs = df_obs[df_obs.annotated_specificity.isin(['OVA','GP33','M45'])].reset_index(drop=True).copy()
df_obs.to_excel('./processed_tcr_data_new_sc_processing/TCR_obs_data_busch_lab_ova_gp33_m45_all_tcrs_reactive.xlsx')
df_obs.to_csv('./processed_tcr_data_new_sc_processing/TCR_obs_data_busch_lab_ova_gp33_m45_all_tcrs_reactive.csv')